# Big Dan's Car Wash — review & sentiment analysis

6,209 Google Maps reviews scraped across **25 Big Dan's locations** (FL / GA / SC / AL),
Nov 2020 → Jul 2026. This notebook turns the raw star ratings + free text into an
operator-facing read: *where is the reputation strong, where is it a risk, and what
specifically drives the unhappy reviews.*

**Method.** Star rating is the ground-truth label; free text (72% of reviews) is scored
with **VADER** — a rule/lexicon sentiment model built for short social text, so it's
deterministic and needs no training or API. We then mine themes (keyword tagging) and
distinctive complaint terms (log-odds of 1–2★ vs 4–5★ vocabulary).

> Everything recomputes from `final_reviews.csv`; re-run top-to-bottom to refresh.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

CSV = Path("/Users/lakshyatomar/Desktop/OX/final_reviews.csv")

# dataviz reference palette (validated). Roles, not raw hex, in the charts.
C = dict(blue="#2a78d6", aqua="#1baf7a", green="#008300", red="#e34948",
         amber="#eda100", orange="#eb6834", gray="#8a8a86",
         ink="#0b0b0b", muted="#52514e", grid="#e8e8e4", surface="#ffffff")
STAR5 = ["#c2452d", "#e0894a", "#9a9a94", "#4b9e5f", "#0b7a34"]  # 1★→5★ diverging

LEGEND = dict(orientation="h", y=-0.18, x=0.5, xanchor="center",
              font=dict(size=11),
              bgcolor="rgba(255,255,255,0.85)", bordercolor="#cccccc", borderwidth=1)
MARGIN_1AX = dict(l=80, r=50, t=110, b=150)
MARGIN_2AX = dict(l=80, r=100, t=120, b=150)

def style(fig, title, legend=False, margin=None, height=None):
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=13)),
        plot_bgcolor="white", paper_bgcolor="white",
        hovermode="x unified",
        showlegend=legend, legend=LEGEND if legend else None,
        margin=margin or MARGIN_1AX,
        height=height or 480,
    )
    return fig

df = pd.read_csv(CSV)
df = df[~(df["reviewId"].duplicated() & df["reviewId"].notna())].copy()   # dedupe reviews
SMAP = {"Florida":"FL","Georgia":"GA","South Carolina":"SC","Alabama":"AL"}
df["st"] = df["state"].map(SMAP).fillna(df["state"])
df["reviewDate"] = pd.to_datetime(df["reviewDate"], errors="coerce", utc=True)
df["ownerResponseDate"] = pd.to_datetime(df["ownerResponseDate"], errors="coerce", utc=True)
df["responded"] = df["ownerResponseText"].notna()
df["isLocalGuide"] = df["isLocalGuide"].astype(str)
df["has_text"] = df["reviewText"].notna()
df["txt_len"] = df["reviewText"].astype(str).str.len().where(df["has_text"])

# VADER compound sentiment on the reviews that have text
an = SentimentIntensityAnalyzer()
df["sent"] = np.nan
df.loc[df["has_text"], "sent"] = [an.polarity_scores(str(x))["compound"]
                                  for x in df.loc[df["has_text"], "reviewText"]]
LOW = df["rating"] <= 2   # "unhappy" review flag used throughout
print(f"{len(df):,} reviews · {df['site'].nunique()} locations · "
      f"{df['has_text'].sum():,} with text · mean rating {df['rating'].mean():.2f}")

6,209 reviews · 25 locations · 4,469 with text · mean rating 4.70


In [2]:
!pip install vaderSentiment

## 1. The shape of the corpus — a 4.7★ average, but read the tail

Two views: how the 6,209 ratings split across stars, and how review **volume** has grown.

In [ ]:
vc = df["rating"].value_counts().sort_index()
pct = vc / len(df) * 100

yr = df.assign(yr=df["reviewDate"].dt.year).groupby("yr").size()
yr = yr[yr.index <= 2026]

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.42, 0.58],
    subplot_titles=[f"87% are 5★ — mean {df['rating'].mean():.2f} hides the 1★ spike",
                     "Review volume is compounding — mostly recent"],
)

fig.add_trace(go.Bar(
    x=vc.index, y=vc.values, marker_color=STAR5, showlegend=False,
    text=[f"{v:,}<br>{p:.0f}%" for v, p in zip(vc.values, pct.values)],
    textposition="outside",
    customdata=pct.values,
    hovertemplate="<b>%{x}★</b><br>Reviews: %{y:,}<br>Share: %{customdata:.0f}%<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=yr.index.astype(int), y=yr.values, marker_color=C["blue"], showlegend=False,
    text=[f"{v:,}" for v in yr.values], textposition="outside",
    hovertemplate="<b>%{x}</b><br>Reviews: %{y:,}<extra></extra>",
), row=1, col=2)

fig.add_annotation(x=2026, y=yr.loc[2026]*1.12, text="2026 (part-yr)", showarrow=False,
                    font=dict(size=10, color=C["muted"]), row=1, col=2)

fig.update_xaxes(title_text="star rating", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="reviews", title_font=dict(size=11), range=[0, vc.max()*1.25], row=1, col=1)
fig.update_xaxes(title_text="year", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="reviews", title_font=dict(size=11), range=[0, yr.max()*1.25], row=1, col=2)

fig = style(fig, "<b>The shape of the corpus</b><br><sup>Rating distribution and review volume by year</sup>",
            margin=MARGIN_1AX)
fig.show()

**Insights:**
- **The average is a poor summary.** 87% of reviews are 5★ and 4% are 1★ — a classic
  J-shaped review distribution. The 4.7 mean is really "mostly delighted, with a small
  hard core of very angry customers"; the 270 one-star reviews are where the actionable
  signal lives, not the mean.
- **The corpus is dominated by the last 18 months** — 2025 (1,902) and 2026-to-date
  (1,838) are 60% of all reviews, so any trend or theme below is effectively a read on
  *current* operations, not ancient history.
- *Caveat:* 2026 is a partial year (through July) yet already nearly matches full-year
  2025 — review acquisition is accelerating (new locations and/or active solicitation),
  which can itself inflate the 5★ share if prompts target happy customers.

## 2. Does the text agree with the stars? (validating the sentiment score)

Before trusting VADER, check it against the ground-truth star rating. If the text
sentiment climbs monotonically with stars, the score is measuring the right thing — and
the places where they *disagree* become interesting on their own.

In [4]:
g = df[df["has_text"]].groupby("rating")["sent"].agg(["mean","count"])

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.52, 0.48],
    subplot_titles=["Monotonic: VADER tracks the stars (bubble = volume)",
                     "Text/star disagreement is rare"],
)

fig.add_trace(go.Scatter(
    x=g.index, y=g["mean"], mode="lines", line=dict(color=C["blue"], width=2),
    showlegend=False, hoverinfo="skip",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=g.index, y=g["mean"], mode="markers+text", showlegend=False,
    marker=dict(size=(g["count"]/g["count"].max()*46+14), color=STAR5,
                line=dict(color="white", width=1.2)),
    text=[f"{v:+.2f}" for v in g["mean"]], textposition="top center",
    customdata=g["count"],
    hovertemplate="<b>%{x}★</b><br>Mean sentiment: %{y:+.2f}<br>n=%{customdata:,}<extra></extra>",
), row=1, col=1)
fig.add_hline(y=0, line=dict(color=C["muted"], width=1), row=1, col=1)

d = df[df["has_text"]]
cats_n = {
    "5★ but<br>negative text": int(((d["rating"]==5)&(d["sent"]<-.05)).sum()),
    "1–2★ but<br>positive text": int(((d["rating"]<=2)&(d["sent"]>.5)).sum()),
}
cats_n["aligned<br>(rest)"] = int(len(d) - sum(cats_n.values()))
labels, vals = list(cats_n.keys()), list(cats_n.values())
fig.add_trace(go.Bar(
    x=labels, y=vals, marker_color=[C["red"], C["amber"], C["aqua"]], showlegend=False,
    text=[f"{v:,}" for v in vals], textposition="outside",
    hovertemplate="<b>%{x}</b><br>Reviews: %{y:,}<extra></extra>",
), row=1, col=2)

fig.update_xaxes(title_text="star rating", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="mean VADER sentiment", title_font=dict(size=11), range=[-.55, .95], row=1, col=1)
fig.update_yaxes(title_text="reviews", title_font=dict(size=11), type="log", row=1, col=2)

fig = style(fig, "<b>Does the text agree with the stars?</b><br><sup>Mean sentiment by rating, and how often the two disagree</sup>",
            margin=MARGIN_1AX)
fig.show()

**Insights:**
- **VADER is validated on this corpus.** Mean sentiment rises strictly with stars
  (−0.31 → −0.00 → +0.13 → +0.47 → +0.74), so the free-text score is a faithful second
  signal, not noise — worth using where stars alone are too coarse (e.g. separating a
  lukewarm 4★ from an enthusiastic one).
- **2★ reviews are genuinely mixed** (mean sentiment ≈ 0.00): these are the "good wash
  but…" reviews where a specific complaint sits inside otherwise neutral prose — the most
  useful text to read verbatim.
- **Disagreement is rare but real:** ~35 five-star reviews carry negative text (sarcasm,
  or praise-then-gripe) and ~47 low-star reviews read positive (a compliment wrapped
  around a billing complaint). Small enough to ignore in aggregate, but they're exactly
  the reviews an auto-rating dashboard would misclassify.
- *Caveat:* VADER misreads domain sarcasm and negation-heavy text; treat the compound
  score as a ranking aid, not a verdict on any single review.

## 3. Location leaderboard — who is a reputation risk?

The brand average hides wide spread across the 25 stores. Ranked by average rating
(locations with ≥30 reviews), colored by whether they clear the brand's 4.7 bar.

In [5]:
site = (df.groupby("site")
        .agg(n=("rating","size"), avg=("rating","mean"), sent=("sent","mean"),
             pct_low=("rating", lambda s:(s<=2).mean()*100),
             resp=("responded","mean"))
        .query("n>=30").sort_values("avg", ascending=False))   # best at top
brand = df["rating"].mean()
colors = [C["red"] if a < 4.6 else C["amber"] if a < brand else C["aqua"] for a in site["avg"]]
names = [s.replace("Big Dan's ", "") for s in site.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=site["avg"], y=names, orientation="h", marker_color=colors, showlegend=False,
    text=[f"{v:.2f}" for v in site["avg"]], textposition="inside", insidetextanchor="end",
    textfont=dict(color="white", size=11),
    customdata=np.stack([site["n"], site["pct_low"], site["sent"]], axis=-1),
    hovertemplate="<b>%{y}</b><br>Avg rating: %{x:.2f}<br>n=%{customdata[0]:,.0f}<br>"
                  "1–2★ share: %{customdata[1]:.1f}%<br>Sentiment: %{customdata[2]:+.2f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[4.02]*len(site), y=names, mode="text", text=[f"n={int(v)}" for v in site["n"]],
    textposition="middle right", textfont=dict(size=10, color=C["muted"]),
    showlegend=False, hoverinfo="skip",
))
fig.add_vline(x=brand, line=dict(color=C["ink"], width=1.2, dash="dash"),
              annotation_text=f"brand avg {brand:.2f}", annotation_position="top",
              annotation_font=dict(size=10, color=C["ink"]))

fig.update_xaxes(title_text="average star rating", range=[4.0, 5.05])
fig.update_yaxes(title_text=None, tickfont=dict(size=10.5))
fig = style(fig, "<b>Location leaderboard — who is a reputation risk?</b><br>"
                  "<sup>Fairburn &amp; Tarpon Springs lag; Academy / Decatur / Kissimmee lead</sup>",
            margin=MARGIN_1AX, height=750)
fig.show()
site.assign(avg=site["avg"].round(2), sent=site["sent"].round(2),
            pct_low=site["pct_low"].round(1), resp=(site["resp"]*100).round(0)).reset_index()

,site,n,avg,sent,pct_low,resp
0,Big Dan's Academy,105,5.00,0.75,0.0,0.0
1,Big Dan's Decatur,115,4.92,0.70,0.9,96.0
2,Big Dan's Kissimmee OBT,93,4.90,0.69,2.2,0.0
3,Big Dan's Columbia,135,4.88,0.73,1.5,95.0
4,Big Dan's John Young Pkwy,107,4.83,0.51,3.7,94.0
5,Big Dan's Fountain Inn,66,4.80,0.71,3.0,0.0
6,Big Dan's St Pete,96,4.80,0.78,4.2,0.0
7,Big Dan's OBT,161,4.80,0.52,4.3,96.0
8,Big Dan's Lady Lake,495,4.79,0.68,3.8,97.0
9,Big Dan's Bradenton 14th St,225,4.79,0.62,4.0,96.0


**Insights:**
- **Spread is half a star — operationally huge.** From **Fairburn 4.44** to **Academy
  5.00** (perfect over n=105). At review-driven-demand scale a 0.5★ gap is the difference
  between ranking first and third in a local map pack.
- **Two clear risk stores.** Fairburn (4.44, **10.2% of reviews are 1–2★**) and Tarpon
  Springs (4.59 but on n=395 — its 8% low-star rate affects the most customers by volume).
  These are where an operator visit pays off, not the already-perfect stores.
- **The leaders aren't the highest-volume stores** — Academy/Decatur/Kissimmee are
  mid-volume, so their near-perfect scores are plausibly *real* (not just thin-sample
  luck), but confirm before celebrating: Acworth's 4.58 sits on only n=33.
- *Caveat:* one store (of 25) falls below the n≥30 cutoff and is excluded; ratings here
  are lifetime, so a store that fixed its issues a year ago still carries its old 1★s.

## 4. What customers actually talk about — themes vs. toxicity

Keyword-tag every text review into themes, then plot each theme by **how often** it comes
up (x) against **how often it turns up in a 1–2★ review** (y). Top-left = frequent &
happy (a strength); bottom-right = rare but toxic (a risk). Bubble size = mentions.

In [6]:
THEMES = {
 "staff / service": r"\b(staff|employee|guy|guys|team|worker|manager|friendly|rude|attitude|helpful|service)\b",
 "quality / clean": r"\b(clean|dirty|spotless|streak|spot|soap|residue|shine|wax|towel)\b",
 "price / value":   r"\b(price|expensive|cheap|value|worth|money|cost|membership|subscription|unlimited|monthly|plan)\b",
 "equipment / vac": r"\b(vacuum|vacuums|brush|machine|dryer|blower|mat)\b",
 "speed / wait":    r"\b(fast|quick|wait|waiting|line|slow|minutes|long)\b",
 "damage":          r"\b(damage|scratch|scratched|broke|broken|dent|cracked|antenna|mirror)\b",
}
low_txt = df["reviewText"].astype(str).str.lower()
rows = []
for name, pat in THEMES.items():
    m = low_txt.str.contains(pat, regex=True, na=False) & df["has_text"]
    rows.append(dict(theme=name, n=int(m.sum()),
                     pct_text=m.sum()/df["has_text"].sum()*100,
                     pct_low=(df.loc[m,"rating"]<=2).mean()*100,
                     sent=df.loc[m,"sent"].mean()))
T = pd.DataFrame(rows)

base_low = (df.loc[df["has_text"],"rating"]<=2).mean()*100
sizes = T["n"]/T["n"].max()*80 + 22
col = [C["red"] if p > base_low*1.4 else C["aqua"] if p < base_low*.8 else C["amber"] for p in T["pct_low"]]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=T["pct_text"], y=T["pct_low"], mode="markers+text", showlegend=False,
    marker=dict(size=sizes, color=col, opacity=.78, line=dict(color="white", width=1.5)),
    text=[f"<b>{r.theme}</b><br>({r.n:,} mentions)" for r in T.itertuples()],
    textposition="middle center", textfont=dict(size=9.5, color=C["ink"]),
    customdata=np.stack([T["n"], T["sent"]], axis=-1),
    hovertemplate="<b>%{text}</b><br>Mentioned in: %{x:.1f}% of text reviews<br>"
                  "1–2★ share: %{y:.1f}%<br>Sentiment: %{customdata[1]:+.2f}<extra></extra>",
))
fig.add_hline(y=base_low, line=dict(color=C["muted"], width=1.2, dash="dot"),
              annotation_text=f"corpus avg {base_low:.1f}% low-star", annotation_position="top left",
              annotation_font=dict(size=10, color=C["muted"]))

fig.update_xaxes(title_text="% of text reviews mentioning the theme", range=[0, 52])
fig.update_yaxes(title_text="% of those mentions that are 1–2★", title_font=dict(size=11), range=[-4, 70])
fig = style(fig, "<b>What customers actually talk about</b><br>"
                  "<sup>Staff is the frequent strength; DAMAGE is rare but radioactive</sup>",
            margin=MARGIN_1AX, height=560)
fig.show()
T.round({"pct_text":1,"pct_low":1,"sent":3})

,theme,n,pct_text,pct_low,sent
0,staff / service,1980,44.3,6.5,0.751
1,quality / clean,966,21.6,9.4,0.727
2,price / value,733,16.4,16.1,0.636
3,equipment / vac,620,13.9,9.8,0.734
4,speed / wait,293,6.6,12.6,0.619
5,damage,89,2.0,62.9,-0.012


**Insights:**
- **Staff is the engine of the brand's reputation.** Mentioned in **44%** of all text
  reviews and overwhelmingly positive (sentiment +0.75, only 6.5% low-star) — Big Dan's
  wins on people, and named-employee praise ("Kenny…") is common. Protect and staff-train
  as the #1 lever.
- **Damage is the reputation killer.** Only ~2% of reviews mention it, but **63% of those
  are 1–2★** (mean sentiment −0.01) — an order of magnitude above the 7% corpus baseline.
  Each damage claim is worth ~9 ordinary complaints in star terms; a clean damage-
  resolution process would move the average more than any marketing.
- **Price/value is the mid-tier friction** — 16% mention it and 16% of those go low-star,
  the highest of the non-damage themes, and it's tangled with membership billing (§5).
- *Caveat:* keyword tagging is lexical, so it over-counts (a review can hit "clean" while
  complaining) and misses paraphrase; the direction (damage ≫ toxic) is robust to that,
  the exact percentages are ±a few points.

## 5. What drives the 1–2★ reviews — the words unhappy customers use

Log-odds of each term appearing in a 1–2★ review vs a 4–5★ review (add-½ smoothing).
High bars = words that show up almost *only* in angry reviews — the complaint fingerprint.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
t = df[df["has_text"]]
pos = t.loc[t["rating"]>=4, "reviewText"].astype(str)
neg = t.loc[t["rating"]<=2, "reviewText"].astype(str)
cv = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=6, max_features=3000)
X = cv.fit_transform(pd.concat([pos, neg])); vocab = np.array(cv.get_feature_names_out())
Xp = np.asarray(X[:len(pos)].sum(0)).ravel(); Xn = np.asarray(X[len(pos):].sum(0)).ravel()
a = .5
lo = (np.log((Xn+a)/(Xn.sum()+a*len(vocab))) - np.log((Xp+a)/(Xp.sum()+a*len(vocab))))
top = np.argsort(lo)[::-1][:16][::-1]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=lo[top], y=vocab[top], orientation="h", marker_color=C["red"], showlegend=False,
    customdata=np.stack([Xn[top], Xp[top]], axis=-1),
    hovertemplate="<b>%{y}</b><br>Log-odds: %{x:.2f}<br>%{customdata[0]}× in 1–2★ · "
                  "%{customdata[1]}× in 4–5★<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=lo[top] + lo[top].max()*.04, y=vocab[top], mode="text",
    text=[f"{Xn[i]}× in 1–2★ · {Xp[i]}× in 4–5★" for i in top],
    textposition="middle right", textfont=dict(size=9, color=C["muted"]),
    showlegend=False, hoverinfo="skip",
))

fig.update_xaxes(title_text="log-odds (higher = more exclusive to angry reviews)",
                  range=[0, lo[top].max()*1.6])
fig.update_yaxes(title_text=None, tickfont=dict(size=10.5))
fig = style(fig, "<b>What drives the 1–2★ reviews</b><br>"
                  "<sup>Complaints cluster on damage-disputes and membership billing</sup>",
            margin=MARGIN_1AX, height=620)
fig.show()

**Insights:**
- **Two complaint clusters, both operational, not about the wash itself.** (1)
  *Damage-dispute*: `ripped`, `got home`, `video`, `denied`, `responsibility`,
  `accountability`, `unprofessional` — customers alleging damage and, worse, feeling
  *stonewalled* on the claim. (2) *Billing*: `charged`, `refund`, `canceled membership` —
  subscription friction.
- **The process, not the product, generates the anger.** `denied`/`accountability`/
  `responsibility` appearing almost exclusively in 1–2★ says the escalation is the
  handling of a claim, not a dirty car. That's fixable with policy (dashcam-friendly
  damage SOP, faster refunds) far more cheaply than equipment.
- **`refund` appears 24× in angry reviews and 0× in happy ones** — a near-perfect
  negative marker. A dashboard that flags any new review containing it would catch the
  costliest reputation events in near-real-time.
- *Caveat:* n=327 low-star text reviews, so individual bigram counts are small (6–25);
  read these as *themes to investigate*, not precisely-ranked causes, and note membership
  billing complaints tie back to the price/value theme in §4.

## 6. Is the reputation improving or slipping? (one axis at a time)

Quarterly. Volume and rating are different scales, so — never a dual axis — they get
two stacked panels sharing the time axis.

In [8]:
q = (df.assign(q=df["reviewDate"].dt.to_period("Q"))
     .groupby("q").agg(n=("rating","size"), avg=("rating","mean"),
                       pct_low=("rating", lambda s:(s<=2).mean()*100)))
q = q[(q.index >= "2021Q3")]      # drop the ~single-review cold-start quarters
x = q.index.astype(str)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=["Review volume by quarter", "Average rating by quarter (dashed = lifetime mean)"],
    row_heights=[0.45, 0.55],
)

fig.add_trace(go.Bar(
    x=x, y=q["n"], marker_color=C["blue"], showlegend=False,
    hovertemplate="<b>%{x}</b><br>Reviews: %{y:,}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x, y=q["avg"], mode="lines+markers", line=dict(color=C["aqua"], width=2),
    marker=dict(size=6), showlegend=False,
    text=[f"{v:.2f}" for v in q["avg"]],
    hovertemplate="<b>%{x}</b><br>Avg rating: %{y:.2f}<extra></extra>",
), row=2, col=1)
fig.add_hline(y=df["rating"].mean(), line=dict(color=C["muted"], width=1, dash="dash"), row=2, col=1)

fig.update_yaxes(title_text="reviews / qtr", title_font=dict(size=11), row=1, col=1)
fig.update_yaxes(title_text="avg rating", title_font=dict(size=11), range=[4.2, 5.02], row=2, col=1)
fig.update_xaxes(tickangle=45, row=2, col=1)

fig = style(fig, "<b>Is the reputation improving or slipping?</b><br>"
                  "<sup>Quarterly — volume and rating on separate axes, never dual-axis</sup>",
            margin=MARGIN_1AX, height=650)
fig.show()

**Insights:**
- **Reputation is stable-to-improving while scaling fast.** Average rating holds in a
  tight **4.55–4.76** band across five years even as quarterly volume grew from double
  digits to ~900 — the operation isn't degrading under growth, which is the failure mode
  you'd most worry about.
- **The two most recent quarters are the highest-rated** — 2026 is running above the
  lifetime mean, so whatever changed (staffing, new stores, damage handling) is moving in
  the right direction, not against it.
- *Caveat:* the latest quarter is partial and recency-biased (fresh 5★s land before their
  angry counterparts get written and before disputes escalate), so read the final point
  as provisional; the *level* stability is the durable finding.

## 7. Reviewer type & operator responsiveness

Two behavioral checks: do **Local Guides** (experienced Google reviewers) rate
differently, and does the owner **respond** where it matters most?

In [9]:
lg = df.groupby("isLocalGuide").agg(avg=("rating","mean"), n=("rating","size"),
                                    length=("txt_len","mean")).loc[["False","True"]]
xs = ["regular reviewer", "Local Guide"]

resp = df.groupby("rating")["responded"].mean()*100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Local Guides rate lower, write longer", "Owner replies to ~89% — even 1★"],
)

fig.add_trace(go.Bar(
    x=xs, y=lg["avg"], marker_color=[C["gray"], C["blue"]], showlegend=False,
    text=[f"{r.avg:.2f}<br>n={int(r.n):,}<br>{r.length:.0f} chars" for r in lg.itertuples()],
    textposition="inside", insidetextanchor="middle", textfont=dict(color="white", size=10.5),
    customdata=np.stack([lg["n"], lg["length"]], axis=-1),
    hovertemplate="<b>%{x}</b><br>Avg rating: %{y:.2f}<br>n=%{customdata[0]:,.0f}<br>"
                  "Avg length: %{customdata[1]:.0f} chars<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=resp.index, y=resp.values, marker_color=STAR5, showlegend=False,
    text=[f"{v:.0f}%" for v in resp.values], textposition="inside",
    insidetextanchor="middle", textfont=dict(color="white", size=10.5),
    hovertemplate="<b>%{x}★</b><br>Owner reply rate: %{y:.0f}%<extra></extra>",
), row=1, col=2)

fig.update_yaxes(title_text="avg rating", title_font=dict(size=11), range=[0, 5], row=1, col=1)
fig.update_xaxes(title_text="star rating", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="% of reviews with owner reply", title_font=dict(size=11), range=[0, 100], row=1, col=2)

fig = style(fig, "<b>Reviewer type &amp; operator responsiveness</b><br>"
                  "<sup>Local Guides vs regular reviewers, and reply rate by star</sup>",
            margin=MARGIN_1AX)
fig.show()
resp_time = ((df["ownerResponseDate"]-df["reviewDate"]).dt.total_seconds()/86400)
resp_time = resp_time[resp_time.between(0,365)]
print(f"owner response time: median {resp_time.median():.1f} days · "
      f"{ (resp_time<=1).mean()*100:.0f}% within 24h")

owner response time: median 0.9 days · 53% within 24h


**Insights:**
- **Local Guides are the tougher, higher-effort audience** — they rate **4.59 vs 4.76**
  (regulars) and write ~25% longer reviews (182 vs 146 chars). Their reviews carry more
  weight on Google *and* contain more of the diagnostic text, so they're the segment worth
  reading closely even though they drag the average.
- **Operator responsiveness is a genuine strength** — ~89% of reviews get an owner reply,
  and critically the response rate barely dips for 1★ (82%): they aren't hiding from
  criticism. Median response time is under a day. This is already best-practice; the gap
  to close is the ~18% of 1★ reviews that go unanswered, which are the highest-value ones.
- *Caveat:* `isLocalGuide` is missing on ~8% of rows (dropped from this split), and a
  reply existing says nothing about whether it *resolved* the issue — §5's `denied`/
  `accountability` terms suggest some replies close the ticket without satisfying the
  customer.

## Executive summary

| Finding | Number | Action |
|---|---|---|
| Reputation runs on **staff** | 44% of reviews, +0.75 sentiment | protect/train; it's the moat |
| **Damage claims** are radioactive | 2% of reviews, **63% are 1–2★** | fix the damage-resolution SOP |
| Complaints are about **process, not the wash** | `refund`/`denied`/`accountability` near-exclusive to 1–2★ | faster refunds, dashcam-friendly claims |
| **Membership billing** friction | price/value theme 16% low-star | simplify cancel/refund |
| Two **risk stores** | Fairburn 4.44, Tarpon Springs 4.59 (n=395) | targeted operator visits |
| Trend is **stable-to-up while scaling** | 4.55→4.76, volume ×20 | keep going |
| Owner response is **strong** | ~89%, <1 day median | close the last 18% of 1★ |

The one-line read: **Big Dan's wins on people and loses on damage-claim handling.** The
average is healthy and improving, but the cheap, high-leverage fix is the process around
the ~2% of interactions that involve alleged damage or a billing dispute — those generate
almost all the reputational downside.

*Data: `final_reviews.csv` (6,209 reviews, deduped). Sentiment: VADER compound. Distinctive
terms: log-odds with add-½ smoothing. Rebuild: `python experiments/reviews/build_reviews_notebook.py`.*